In [1]:
import sys
new_path = '/data/Pangu_RLHF/bowen/new_verl/verl2/verl'
sys.path.append(new_path)

In [2]:
from verl.workers.rollout.vllm_rollout.vllm_rollout_spmd import vLLMRollout
from verl import DataProto
import torch

batch = DataProto.from_single_dict({
        'input_ids': torch.tensor([[1, 2, 3, 4, 5]]),
        'tgt_input_ids': torch.tensor([[1, 2, 3, 4, 5]])
    })
idx = batch.batch['input_ids']
tgt_input_ids = batch.batch['tgt_input_ids']

batch_size = tgt_input_ids.size(0)
batch_size

/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-20 18:16:59,731	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10/site-packages/torch_npu/__init__.py:298: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


1

In [2]:
import sys
sys.path.insert(0, '/data/Pangu_RLHF/bowen/new_verl/verl2')
import verl
print(verl.__file__)  # 应该显示正确的路径了

/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-20 18:27:19,130	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10/site-packages/torch_npu/__init__.py:298: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


/data/Pangu_RLHF/bowen/new_verl/verl2/verl/__init__.py


In [10]:
import sys
print(sys.path)

['/data/Pangu_RLHF/bowen/new_verl/verl2', '/data/Pangu_RLHF/bowen/new_verl/verl2/verl', '/data/Pangu_RLHF/bowen/new_verl/verl2/verl', '/data/Pangu_RLHF/bowen/new_verl/verl2', '/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10/site-packages/ray/thirdparty_files', '/data/Pangu_RLHF/bowen/new_verl/verl2/verl/workers/rollout/vllm_rollout', '/usr/local/Ascend/ascend-toolkit/latest/tools/ms_fmk_transplt/torch_npu_bridge', '', '/usr/local/Ascend/ascend-toolkit/latest/python/site-packages', '/usr/local/seccomponent/lib', '/home/ma-user/infer/model/1', '/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python310.zip', '/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10', '/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10/lib-dynload', '/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10/site-packages', '/data/Pangu_RLHF/bowen/verl', '/home/ma-user/modelarts-dev/modelarts-sdk', '/home/ma-user/modelarts-dev/ma-cli', '/modelarts/tools/solution/advisor', '/modelarts/tools/

In [11]:
import pkg_resources
try:
    verl_dist = pkg_resources.get_distribution('verl')
    print(f"Installed verl location: {verl_dist.location}")
    print(f"Installed verl version: {verl_dist.version}")
except:
    print("verl not installed as package")

Installed verl location: /home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.10/site-packages
Installed verl version: 0.5.0.dev0


In [8]:
from typing import List
def _pre_process_inputs(pad_token_id, prompt_token_ids: torch.Tensor) -> List[int]:
    # remove the left padding in the prompt token_id
    # pad_token_id = self.llm_engine.tokenizer.pad_token_id if self.llm_engine.tokenizer.pad_token_id is not None else self.llm_engine.tokenizer.eos_token_id
    non_pad_index = torch.nonzero(prompt_token_ids != pad_token_id, as_tuple=False)[0][0]
    token_ids = prompt_token_ids[non_pad_index:].tolist()
    return token_ids

In [12]:
# idx_list = [1, 2, 3, 4, 5]
idx_list = [
    _pre_process_inputs(1, idx[i])
    for i in range(batch_size)
]
idx_list = sum([[idx_list[i]] * 2 for i in range(len(idx_list))], [])

idx_list


[[2, 3, 4, 5], [2, 3, 4, 5]]

In [13]:
tgt_input_ids = batch.batch['tgt_input_ids']  # [bsz, tgt_len]

tgt_list = [
    _pre_process_inputs(1, tgt_input_ids[i])
    for i in range(batch_size)
]

In [20]:
tgt_list = sum([[tgt_list[i]] * 2 for i in range(len(tgt_list))], [])
import random
prefix_ratios = [random.randint(0, 100)/100 for _ in range(len(tgt_list))]
prefix_list = [tgt_list[i][:int(len(tgt_list[i]) * prefix_ratios[i])] for i in range(len(tgt_list))]
idx_list = [idx_list[i] + prefix_list[i] for i in range(len(idx_list))]
print(idx_list)
print(tgt_list)
print(prefix_list)

[[2, 3, 4, 5], [2, 3, 4, 5, 2, 3]]
[[2, 3, 4, 5], [2, 3, 4, 5], [2, 3, 4, 5], [2, 3, 4, 5], [2, 3, 4, 5], [2, 3, 4, 5], [2, 3, 4, 5], [2, 3, 4, 5]]
[[], [2, 3], [], [2], [2, 3], [2, 3, 4], [2], [2]]


In [21]:
prefix_ratios

[0.23, 0.61, 0.0, 0.28, 0.72, 0.97, 0.31, 0.3]

In [22]:
prefix_list

[[], [2, 3], [], [2], [2, 3], [2, 3, 4], [2], [2]]

In [3]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('/home/ma-user/work/dingbowen/model_zoo/Qwen2.5-0.5B-Instruct')

In [10]:
prompt_token_ids=[151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 565, 5430, 220, 20, 271, 12275, 13, 98918, 702, 27548, 220, 16, 21, 10496, 14201, 382, 4340, 1657, 25904, 646, 566, 1936, 30, 151645, 198, 151644, 77091, 198, 13708, 766, 397, 32313, 11, 773, 4392, 13, 98918, 1865, 220, 16, 21, 10496, 14201, 13, 576, 3405, 374, 10161, 1246, 1657, 25904, 566, 646, 1936, 448, 1846, 14201, 13, 88190, 11, 1077, 594, 1490, 13, 358, 1184, 311, 7071, 700, 1246, 1657, 14201, 1817, 10496, 7460, 382, 3830, 1128, 358, 1414, 11, 1429, 25904, 614, 3040, 14201, 1817, 13, 8909, 11, 264, 5297, 10496, 11, 1290, 30, 1446, 1414, 11, 825, 518, 1817, 9131, 13, 1988, 3783, 11, 7196, 1045, 25904, 614, 2326, 14201, 30, 2521, 7196, 2155, 14431, 30, 1988, 279, 3491, 3171, 944, 13837, 894, 3281, 3093, 315, 10496, 13, 1084, 1101, 2727, 330, 331, 4720, 1335, 773, 358, 1265, 4658, 9658, 279, 5297, 3040, 94264, 10496, 382, 2679, 1817, 10496, 3880, 3040, 14201, 11, 1221, 279, 1372, 315, 25904, 566, 646, 1936, 1035, 387, 279, 2790, 1372, 315, 14201, 17779, 553, 14201, 817, 10496, 13, 2055, 11, 220, 16, 21, 17779, 553, 220, 19, 13, 6771, 752, 653, 429, 21937, 13, 220, 16, 21, 17779, 553, 220, 19, 374, 220, 19, 13, 2055, 11, 429, 1035, 3076, 566, 646, 1936, 220, 19, 25904, 382, 3983, 3783, 11, 1077, 752, 1990, 15934, 13, 2160, 1052, 894, 13038, 429, 25904, 2578, 1373, 803, 476, 16745, 14201, 30, 8909, 11, 7196, 1045, 89772, 614, 2326, 14201, 11, 714, 1549, 11, 279, 3491, 3171, 944, 6286, 89772, 13, 1084, 11689, 2727, 25904, 13, 7281, 11, 7025, 25904, 2578, 614, 803, 14201, 369, 19753, 11, 714, 11136, 11, 3040, 374, 5297, 13, 2055, 7241, 1052, 594, 264, 14068, 1588, 11, 1075, 7196, 1045, 14201, 525, 10865, 476, 566, 3880, 311, 990, 4960, 14201, 369, 2494, 770, 11, 714, 279, 3491, 3171, 944, 1584, 429, 13, 1084, 1101, 2727, 566, 702, 27548, 220, 16, 21, 10496, 14201, 13, 2055, 358, 1744, 279, 30339, 4226, 374, 220, 16, 21, 17779, 553, 220, 19, 11, 892, 374, 220, 19, 25904, 382, 54815, 11, 279, 4226, 1265, 387, 220, 19, 624, 522, 26865, 397, 27, 9217, 1339, 12275, 13, 98918, 702, 27548, 220, 16, 21, 10496, 14201, 13, 8704, 1817, 5297, 10496, 7460, 220, 19, 14201, 11, 279, 1372, 315, 25904, 566, 646, 1936, 374, 16588, 553, 49702, 279, 2790, 1372, 315, 14201, 553, 279, 1372, 4362, 817, 10496, 1447, 59, 9640, 59, 37018, 90, 16, 21, 1124, 1318, 90, 14201, 3417, 90, 19, 1124, 1318, 90, 14201, 817, 10496, 3417, 284, 220, 19, 1124, 1318, 90, 25904, 532, 59, 2533, 334, 16141, 66963, 1124, 79075, 90, 19, 5361, 9217, 29, 151645]

In [7]:
res=(1249, 8253, 1246, 1657, 6785, 25780, 537, 47905, 17767, 16, 15, 61, 21, 57758, 525, 74916, 553, 518, 3245, 825, 315, 220, 18, 11, 220, 20, 11, 476, 220, 22, 11, 582, 1184, 311, 1156, 11047, 279, 2790, 1760, 315, 25780, 74916, 553, 1493, 5109, 11, 323, 1221, 32256, 279, 5109, 429, 525, 74916, 553, 678, 2326, 315, 1105, 13, 5692, 525, 279, 7354, 1447, 16, 13, 3070, 2507, 279, 25780, 74916, 553, 220, 18, 25, 1019, 256, 481, 576, 7772, 7546, 537, 47905, 17767, 16, 15, 61, 21, 57758, 429, 374, 74916, 553, 220, 18, 374, 17767, 16, 15, 61, 21, 608, 220, 18, 284, 220, 18, 18, 18, 18, 18, 18, 59, 4292, 256, 481, 576, 1372, 315, 25780, 504, 220, 16, 311, 17767, 18, 18, 18, 18, 18, 18, 57758, 429, 525, 74916, 553, 220, 18, 374, 1124, 11520, 2359, 59, 11008, 3993, 1124, 37018, 90, 16, 15, 61, 21, 15170, 18, 92, 1124, 1291, 59, 8052, 3993, 284, 220, 18, 18, 18, 18, 18, 18, 1124, 15136, 1124, 37018, 90, 16, 15, 61, 21, 15170, 18, 92, 284, 220, 16, 16, 16, 16, 16, 16, 1124, 15136, 220, 16, 15, 61, 21, 284, 220, 16, 16, 16, 16, 16, 16, 15, 15, 15, 15, 284, 220, 24, 16, 21, 17, 23, 20, 22, 21, 59, 3593, 17, 13, 3070, 2507, 279, 25780, 74916, 553, 220, 20, 25, 1019, 256, 481, 576, 7772, 7546, 537, 47905, 17767, 16, 15, 61, 21, 57758, 429, 374, 74916, 553, 220, 20, 374, 17767, 16, 15, 61, 21, 608, 220, 20, 284, 220, 17, 15, 15, 15, 15, 15, 59, 4292, 256, 481, 576, 1372, 315, 25780, 504, 220, 16, 311, 17767, 17, 15, 15, 15, 15, 15, 57758, 429, 525, 74916, 553, 220, 20, 374, 1124, 11520, 2359, 59, 11008, 3993, 1124, 37018, 90, 16, 15, 61, 21, 15170, 20, 92, 1124, 1291, 59, 8052, 3993, 284, 220, 17, 15, 15, 15, 15, 15, 1124, 15136, 1124, 37018, 90, 16, 15, 61, 21, 15170, 20, 92, 284, 220, 17, 15, 15, 15, 15, 15, 1124, 15136, 220, 17, 15, 15, 15, 15, 15, 284, 220, 19, 15, 15, 15, 15, 15, 15, 15, 15, 15, 284, 220, 18, 24, 24, 22, 19, 24, 23, 24, 22, 21, 59, 3593, 18, 13, 3070, 2507, 279, 25780, 74916, 553, 220, 22, 25, 1019, 256, 481, 576, 7772, 7546, 537, 47905, 17767, 16, 15, 61, 21, 57758, 429, 374, 74916, 553, 220, 22, 374, 17767, 16, 15, 61, 21, 608, 220, 22, 284, 220, 16, 19, 17, 23, 20, 22, 59, 4292, 256, 481, 576, 1372, 315, 25780, 504, 220, 16, 311, 17767, 16, 19, 17, 23, 20, 22, 57758, 429, 525, 74916, 553, 220, 22, 374, 1124, 11520, 2359, 59, 11008, 3993, 1124, 37018, 90, 16, 15, 61, 21, 15170, 22, 92, 1124, 1291, 59, 8052, 3993, 284, 220, 16, 19, 17, 23, 20, 22, 1124, 15136, 1124, 37018, 90, 16, 15, 61, 21, 15170, 22, 92, 284, 220, 17, 15, 18, 17, 17, 21, 15, 1124, 15136, 220, 16, 15, 61, 21, 284, 220, 17, 15, 18, 17, 17, 21, 15, 15, 15, 15, 15, 15, 284, 220, 16, 23, 17, 17, 17, 22, 20, 22, 16, 22, 21, 59, 3593, 19, 13, 3070, 2507, 279, 25780, 74916, 553, 220, 18, 323, 220, 20, 320, 72, 1734, 2572, 220, 16, 20, 1648, 1019, 256, 481, 576, 7772, 7546, 537, 47905, 17767, 16, 15, 61, 21, 57758, 429, 374, 74916, 553, 220, 16, 20, 374, 17767, 16, 15, 61, 21, 608, 220, 16, 20, 284, 220, 21, 21, 21, 21, 21, 59, 4292, 256, 481, 576, 1372, 315, 25780, 504, 220, 16, 311, 17767, 21, 21, 21, 21, 21, 57758, 429, 525, 74916, 553, 220, 16, 20, 374, 1124, 11520, 2359, 59, 11008, 3993, 1124, 37018, 90, 16, 15, 61, 21, 15170, 16, 20, 92, 1124, 1291, 59, 8052, 3993, 284, 220, 21, 21, 21, 21, 21, 1124, 15136, 1124, 37018, 90, 16, 15, 61, 21, 15170, 16, 20, 92, 284, 220, 19, 19, 19, 19, 19, 1124, 15136, 220, 16, 15, 61, 21, 284, 220, 19, 19, 19, 19, 19, 15, 15, 15, 15, 15, 15, 284, 220, 19, 19, 19, 19, 19, 15, 15, 15, 15, 15, 15, 59, 3593, 20, 13, 3070, 2507, 279, 25780, 74916, 553, 220, 18, 11, 220, 20, 11, 323, 220, 22, 320, 72, 1734, 2572, 220, 16, 15, 20, 1648, 1019, 256, 481, 576, 7772, 7546, 537, 47905, 17767, 16, 15, 61, 21, 57758, 429, 374, 74916, 553, 220, 16, 15, 20, 374, 17767, 16, 15, 61, 21, 608, 220, 16, 15, 20, 284, 220, 24, 15, 24, 15, 24, 59, 4292, 256, 481, 576, 1372, 315, 25780, 504, 220, 16, 311, 17767, 24, 15, 24, 15, 24, 57758, 429, 525, 74916, 553, 220, 16, 15, 20, 374, 1124, 11520, 2359, 59, 11008, 3993, 1124, 37018, 90, 16, 15, 61, 21, 15170, 16, 15, 20, 92, 1124, 1291, 59, 8052, 3993, 284, 220, 24, 15, 24, 15, 24, 1124, 15136, 1124, 37018, 90, 16, 15, 61, 21, 15170, 16, 15, 20, 92, 284, 220, 23, 22, 17, 22, 1124, 15136, 220, 16, 15, 61, 21, 284, 220, 23, 22, 17, 22, 15, 15, 15, 15, 15, 15, 15, 284, 220, 23, 22, 17, 22, 15, 15, 15, 15, 15, 15, 15, 59, 3593, 21, 13, 3070, 47866, 279, 1372, 315, 25780, 74916, 553, 518, 3245, 825, 315, 220, 18, 11, 220, 20, 11, 476, 220, 22, 25, 1019, 256, 481, 93210, 279, 5109, 74916, 553, 220, 16, 20, 323, 220, 16, 15, 20, 504, 279, 1760, 315, 5109, 74916, 553, 220, 18, 11, 220, 20, 11, 323, 220, 22, 11, 1221, 912, 279, 1372, 220, 16, 15, 15, 537, 5435, 311, 279, 330, 269, 1, 5666, 320, 11284, 432, 594, 5230, 304, 279, 1760, 315, 5109, 74916, 553, 518, 3245, 825, 315, 220, 18, 11, 220, 20, 11, 476, 220, 22, 4292, 256, 481, 10657, 1760, 1573, 17592, 220, 16, 15, 15, 374, 17767, 24, 16, 21, 17, 23, 20, 22, 21, 488, 220, 18, 24, 24, 22, 19, 24, 23, 24, 22, 21, 488, 220, 16, 23)

In [11]:
print(tokenizer.decode(prompt_token_ids))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
## Task 5

Mr. Müller has manufactured 16 chair legs.

How many chairs can he build?<|im_end|>
<|im_start|>assistant
<think>
Okay, so Mr. Müller made 16 chair legs. The question is asking how many chairs he can build with those legs. Hmm, let's see. I need to figure out how many legs each chair requires.

From what I know, most chairs have four legs each. Like, a standard chair, right? You know, one at each corner. But wait, maybe some chairs have three legs? Or maybe different designs? But the problem doesn't specify any special kind of chair. It just says "chairs," so I should probably assume the standard four-legged chair.

If each chair needs four legs, then the number of chairs he can build would be the total number of legs divided by legs per chair. So, 16 divided by 4. Let me do that calculation. 16 divided by 4 is 4. So, that would mean he can build 4 chairs.

But 

In [9]:
print(tokenizer.decode((198, 151643)))


<|endoftext|>


In [13]:
tokenizer.decode(tokenizer.pad_token_id)

'<|endoftext|>'

In [2]:
import pandas as pd
openr1_220k = pd.read_parquet('/data/Pangu_RLHF/bowen/new_verl/data_zoo/rl_data/openr1.parquet')
openr1_220k

,data_source,prompt,target,ability,reward_model,extra_info
0,olympiads,[{'content': '## Task B-1.3. A ship traveling...,"[{'content': '<think> Okay, so I need to find ...",,"{'ground_truth': 'v_{R}=4\mathrm{~}/\mathrm{},...","{'index': -1, 'split': 'default'}"
1,olympiads,[{'content': '3. (6 points) A construction com...,"[{'content': '<think> Okay, let me try to figu...",,"{'ground_truth': '180', 'style': 'rule'}","{'index': -1, 'split': 'default'}"
2,cn_contest,[{'content': '4. Given the three sides of an o...,"[{'content': '<think> Okay, so I have this pro...",,"{'ground_truth': 'D', 'style': 'rule'}","{'index': -1, 'split': 'default'}"
3,olympiads,[{'content': '1. Solve the equation: $\frac{8 ...,"[{'content': '<think> Okay, so I need to solve...",,"{'ground_truth': '\frac{1}{4}', 'style': 'rule'}","{'index': -1, 'split': 'default'}"
4,aops_forum,[{'content': 'Let $a_n\ (n\geq 1)$ be the valu...,"[{'content': '<think> Okay, so I need to find ...",,"{'ground_truth': '-\ln 2', 'style': 'rule'}","{'index': -1, 'split': 'default'}"
...,...,...,...,...,...,...
45787,olympiads,"[{'content': '4. For any real numbers $x, y$, ...","[{'content': '<think> Okay, so I need to find ...",,"{'ground_truth': '4', 'style': 'rule'}","{'index': -1, 'split': 'default'}"
45788,olympiads,"[{'content': '8.6. On a certain segment, its e...","[{'content': '<think> Okay, so I need to figur...",,"{'ground_truth': '11', 'style': 'rule'}","{'index': -1, 'split': 'default'}"
45789,olympiads,[{'content': '3.075. $(\cos \alpha-\cos 2 \bet...,"[{'content': '<think> Okay, so I need to simpl...",,{'ground_truth': '4\sin^{2}\frac{\alpha+2\beta...,"{'index': -1, 'split': 'default'}"
45790,olympiads,[{'content': '385. $(\tan \alpha+\cot \alpha)^...,"[{'content': '<think> Okay, so I have this exp...",,"{'ground_truth': '4', 'style': 'rule'}","{'index': -1, 'split': 'default'}"


In [3]:
import torch
torch.npu.empty_cache()

In [6]:
import torch_npu
def empty_cache():
    r"""Releases all unoccupied cached memory currently held by the caching
    allocator so that those can be used in other NPU application and visible in
    `nvidia-smi`.

    .. note::
        :func:`~torch_npu.npu.empty_cache` doesn't increase the amount of NPU
        memory available for PyTorch. However, it may help reduce fragmentation
        of NPU memory in certain cases. See :ref:`npu-memory-management` for
        more details about NPU memory management.
    """
    torch_npu._C._npu_emptyCache()
empty_cache()

In [7]:
print(torch.npu.is_available())

True


In [8]:
hasattr(torch, 'npu')

True